# 1D CNN drought severity model

This notebook trains a small **1D CNN** for the Kaggle drought severity competition.

Main design:

- Uses **weekly data** instead of daily rows.
- Uses a configurable number of recent years: `N_YEARS_TO_USE`.
- Default is the latest **2 years** per region.
- Uses a Kaggle-like setup: **13 weeks of weather input** + old score history → **5 future weekly scores**.
- Uses GPU automatically if available.
- Creates a Kaggle submission CSV.

In [1]:
import os
import gc
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Settings

Change `N_YEARS_TO_USE` to control how many recent years are used for training.

Set `USE_LAST_N_YEARS = False` to use all available years, but this can be slow.

In [2]:
TRAIN_PATH = "/kaggle/input/datasets/cheukhongtang/drought/train.csv"
TEST_PATH = "/kaggle/input/datasets/cheukhongtang/drought/test.csv"
SUB_PATH = "/kaggle/input/datasets/cheukhongtang/drought/sample_submission.csv"

OUT_DIR = "/kaggle/working/cnn_1d_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

TARGET = "score"
INPUT_WEEKS = 13
HORIZON = 5

USE_LAST_N_YEARS = True
N_YEARS_TO_USE = 2
WEEKS_PER_YEAR = 52
KEEP_LAST_N_WEEKS = int(N_YEARS_TO_USE * WEEKS_PER_YEAR)

VALIDATION_LAST_N_ANCHORS = 5

# Keep the CNN small first. Increase after the notebook runs.
MAX_DAILY_WEATHER_FEATURES = 12
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 1e-3
PATIENCE = 5

RANDOM_STATE = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 2. Load data

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SUB_PATH)

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

train: (12319040, 17)
test: (204568, 16)
sample_submission: (2248, 6)


,region_id,date,prec,surf_pre,humidity,tmp,dp_tmp,wb_tmp,tmp_max,tmp_min,tmp_range,surf_tmp,wind,wind_max,wind_min,wind_range,score
0,R1,3004-12-31,0.00,101.27,3.76,5.89,-0.14,-0.11,14.87,-2.37,17.24,4.23,1.45,2.58,0.24,2.34,NaN
1,R1,3005-01-01,0.00,101.26,5.37,8.81,4.59,4.60,17.28,1.14,16.14,7.96,1.83,2.38,1.18,1.21,NaN
2,R1,3005-01-02,0.01,100.81,9.32,13.09,12.74,12.74,17.38,8.21,9.17,12.96,1.96,2.49,1.39,1.09,NaN
3,R1,3005-01-03,0.02,100.51,11.40,16.01,16.08,16.08,19.24,13.77,5.47,15.82,2.13,2.67,1.74,0.93,NaN
4,R1,3005-01-04,1.93,100.16,12.20,17.98,17.04,17.04,22.85,14.27,8.58,17.86,2.91,4.25,2.13,2.11,NaN


,region_id,date,prec,surf_pre,humidity,tmp,dp_tmp,wb_tmp,tmp_max,tmp_min,tmp_range,surf_tmp,wind,wind_max,wind_min,wind_range
0,R1,3020-09-18,0.0,99.91,13.62,30.05,18.89,18.69,38.72,22.65,16.07,30.37,1.77,3.75,0.40,3.35
1,R1,3020-09-19,0.0,100.32,12.63,27.09,17.78,17.53,34.11,21.74,12.37,27.49,2.99,4.22,1.87,2.35
2,R1,3020-09-20,0.0,100.68,8.68,24.43,11.94,11.85,32.46,18.65,13.80,24.79,2.51,3.33,1.84,1.49
3,R1,3020-09-21,0.0,100.71,10.35,24.94,14.71,14.53,33.41,18.06,15.35,25.33,2.15,3.12,1.63,1.49
4,R1,3020-09-22,0.0,100.49,10.09,25.18,14.26,14.16,33.92,17.54,16.39,25.27,1.81,2.27,1.36,0.91


,region_id,pred_week1,pred_week2,pred_week3,pred_week4,pred_week5
0,R1,0,0,0,0,0
1,R2,0,0,0,0,0
2,R3,0,0,0,0,0
3,R4,0,0,0,0,0
4,R6,0,0,0,0,0


## 3. Safe date parsing

The dataset can contain invalid calendar years, so this notebook does **not** use `pd.to_datetime()`.
It only extracts month and day from the string.

In [4]:
def add_safe_date_parts(df):
    df = df.copy()
    df["row_order"] = np.arange(len(df), dtype=np.int64)

    date_str = df["date"].astype(str)
    extracted = date_str.str.extract(r"(?P<year>-?\d+)-(?P<month>\d{1,2})-(?P<day>\d{1,2})")

    df["year_raw"] = pd.to_numeric(extracted["year"], errors="coerce").astype("float32")
    df["month"] = pd.to_numeric(extracted["month"], errors="coerce").astype("float32")
    df["day"] = pd.to_numeric(extracted["day"], errors="coerce").astype("float32")

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12).astype("float32")
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12).astype("float32")

    return df

train = add_safe_date_parts(train)
test = add_safe_date_parts(test)

display(train[["date", "year_raw", "month", "day", "month_sin", "month_cos"]].head())

,date,year_raw,month,day,month_sin,month_cos
0,3004-12-31,3004.0,12.0,31.0,-2.449294e-16,1.000000
1,3005-01-01,3005.0,1.0,1.0,5.000000e-01,0.866025
2,3005-01-02,3005.0,1.0,2.0,5.000000e-01,0.866025
3,3005-01-03,3005.0,1.0,3.0,5.000000e-01,0.866025
4,3005-01-04,3005.0,1.0,4.0,5.000000e-01,0.866025


## 4. Select daily weather features

To keep the CNN fast, we select only a limited number of meteorological features before weekly aggregation.
You can increase `MAX_DAILY_WEATHER_FEATURES` after confirming the notebook runs.

In [5]:
exclude_cols = [
    "region_id", "date", "row_order", "year_raw", "month", "day",
    "month_sin", "month_cos", TARGET
]

all_daily_weather_features = [
    c for c in train.columns
    if c not in exclude_cols
    and pd.api.types.is_numeric_dtype(train[c])
]

priority_keywords = ["prec", "humidity", "tmp", "wind", "surf_pre", "dp_tmp", "wb_tmp"]

selected_daily_features = []
for key in priority_keywords:
    selected_daily_features += [c for c in all_daily_weather_features if key in c.lower() and c not in selected_daily_features]

selected_daily_features += [c for c in all_daily_weather_features if c not in selected_daily_features]
selected_daily_features = selected_daily_features[:MAX_DAILY_WEATHER_FEATURES]

print("All daily weather features:", len(all_daily_weather_features))
print("Selected daily weather features:", len(selected_daily_features))
print(selected_daily_features)

All daily weather features: 14
Selected daily weather features: 12
['prec', 'humidity', 'tmp', 'dp_tmp', 'wb_tmp', 'tmp_max', 'tmp_min', 'tmp_range', 'surf_tmp', 'wind', 'wind_max', 'wind_min']


## 5. Daily → weekly aggregation

Every 7 consecutive days per region become one weekly row.
For each selected daily weather variable, we compute weekly summaries.

In [6]:
def first_non_na(x):
    x = x.dropna()
    if len(x) == 0:
        return np.nan
    return x.iloc[0]


def make_weekly_data(df, has_score=True):
    df = df.copy()
    df = df.sort_values(["region_id", "row_order"]).reset_index(drop=True)

    df["day_idx_region"] = df.groupby("region_id").cumcount().astype(np.int32)
    df["week_idx"] = (df["day_idx_region"] // 7).astype(np.int32)

    agg_dict = {}
    for col in selected_daily_features:
        agg_dict[col] = ["mean", "std", "min", "max", "sum"]

    agg_dict["month"] = ["first"]
    agg_dict["month_sin"] = ["mean"]
    agg_dict["month_cos"] = ["mean"]

    if has_score:
        agg_dict[TARGET] = first_non_na

    weekly = df.groupby(["region_id", "week_idx"], as_index=False).agg(agg_dict)

    new_cols = []
    for col in weekly.columns:
        if isinstance(col, tuple):
            if col[1] == "":
                new_cols.append(col[0])
            elif col[1] == "first_non_na":
                new_cols.append(col[0])
            else:
                new_cols.append(f"{col[0]}_{col[1]}")
        else:
            new_cols.append(col)

    weekly.columns = new_cols
    weekly = weekly.rename(columns={
        "month_first": "month",
        "month_sin_mean": "month_sin",
        "month_cos_mean": "month_cos",
    })

    float_cols = weekly.select_dtypes(include=["float64"]).columns
    weekly[float_cols] = weekly[float_cols].astype("float32")

    return weekly

weekly_train = make_weekly_data(train, has_score=True)
weekly_test = make_weekly_data(test, has_score=False)

del train, test
gc.collect()

print("weekly_train:", weekly_train.shape)
print("weekly_test:", weekly_test.shape)
print("weeks per test region:")
display(weekly_test.groupby("region_id").size().describe())

display(weekly_train.head())

weekly_train: (1760184, 66)
weekly_test: (29224, 65)
weeks per test region:


count    2248.0
mean       13.0
std         0.0
min        13.0
25%        13.0
50%        13.0
75%        13.0
max        13.0
dtype: float64

,region_id,week_idx,prec_mean,prec_std,prec_min,prec_max,prec_sum,humidity_mean,humidity_std,humidity_min,humidity_max,humidity_sum,tmp_mean,tmp_std,tmp_min,tmp_max,tmp_sum,dp_tmp_mean,dp_tmp_std,dp_tmp_min,dp_tmp_max,dp_tmp_sum,wb_tmp_mean,wb_tmp_std,wb_tmp_min,wb_tmp_max,wb_tmp_sum,tmp_max_mean,tmp_max_std,tmp_max_min,tmp_max_max,tmp_max_sum,tmp_min_mean,tmp_min_std,tmp_min_min,tmp_min_max,tmp_min_sum,tmp_range_mean,tmp_range_std,tmp_range_min,tmp_range_max,tmp_range_sum,surf_tmp_mean,surf_tmp_std,surf_tmp_min,surf_tmp_max,surf_tmp_sum,wind_mean,wind_std,wind_min,wind_max,wind_sum,wind_max_mean,wind_max_std,wind_max_min,wind_max_max,wind_max_sum,wind_min_mean,wind_min_std,wind_min_min,wind_min_max,wind_min_sum,month,month_sin,month_cos,score
0,R1,0,2.520000,5.842682,0.0,15.67,17.639999,7.640000,3.705100,2.80,12.20,53.480000,10.762857,5.830139,1.32,17.98,75.339996,8.154285,8.144376,-3.97,17.040001,57.080002,8.171429,8.121725,-3.90,17.040001,57.200001,16.590000,5.007681,6.66,22.85,116.129997,4.831429,7.439979,-4.29,14.27,33.82,11.758572,4.382835,5.47,17.24,82.309998,10.334286,6.045439,1.41,17.860001,72.339996,2.341429,0.659051,1.45,3.15,16.389999,3.228571,0.877410,2.38,4.25,22.600000,1.528571,0.667244,0.24,2.13,10.70,12.0,0.428571,0.885165,0.0
1,R1,1,1.344286,2.313359,0.0,5.37,9.410000,3.637143,1.424532,1.85,6.16,25.459999,1.952857,3.318226,-2.33,7.83,13.670000,-1.315714,5.169774,-9.02,6.800000,-9.210000,-1.251429,5.091108,-8.75,6.800000,-8.760000,8.304286,4.703310,2.53,15.79,58.130001,-3.240000,3.332962,-7.23,2.86,-22.68,11.542857,3.785863,6.19,16.77,80.800003,1.805714,3.330209,-2.57,7.640000,12.640000,1.887143,0.390799,1.36,2.47,13.210000,2.665714,0.574336,1.92,3.61,18.660000,0.891429,0.610667,0.17,1.79,6.24,1.0,0.500000,0.866025,0.0
2,R1,2,2.222857,4.988960,0.0,13.48,15.560000,5.311429,2.300894,2.63,8.36,37.180000,6.785714,4.701903,-0.33,12.83,47.500000,3.288571,6.521785,-5.00,10.880000,23.020000,3.320000,6.487223,-4.89,10.890000,23.240000,12.370000,3.923961,6.25,16.66,86.589996,0.980000,4.879857,-5.89,6.54,6.86,11.391429,2.432553,8.83,15.59,79.739998,6.785714,4.630658,-0.21,12.750000,47.500000,2.491429,0.604412,1.79,3.39,17.440001,3.391428,0.839115,2.35,4.45,23.740000,1.588571,0.517282,0.75,2.12,11.12,1.0,0.500000,0.866025,0.0
3,R1,3,3.562857,7.238031,0.0,19.66,24.940001,5.712857,2.978812,3.11,10.56,39.990002,7.435714,5.438207,2.23,15.17,52.049999,3.988571,7.184069,-2.73,14.610000,27.920000,4.021429,7.158937,-2.68,14.610000,28.150000,14.125714,4.043335,9.54,18.99,98.879997,1.208571,6.387698,-4.00,10.84,8.46,12.918571,3.007199,8.14,15.74,90.430000,7.350000,5.418905,2.12,15.070000,51.450001,2.447143,0.904870,1.71,3.86,17.129999,3.387143,1.302328,2.36,5.28,23.709999,1.272857,0.646058,0.24,2.07,8.91,1.0,0.500000,0.866025,0.0
4,R1,4,0.341429,0.797129,0.0,2.14,2.390000,3.820000,1.010511,2.25,5.15,26.740000,4.124286,2.083737,0.25,6.42,28.870001,-0.442857,3.804093,-6.68,4.130000,-3.100000,-0.390000,3.751631,-6.53,4.140000,-2.730000,10.007143,1.765519,7.44,11.86,70.050003,-1.382857,3.528861,-6.46,3.69,-9.68,11.387143,3.991085,3.74,16.09,79.709999,3.722857,2.673036,-1.13,6.450000,26.059999,2.328571,0.553577,1.65,3.24,16.299999,3.110000,0.778160,2.36,4.41,21.770000,1.588571,0.404328,0.92,2.14,11.12,1.0,0.656868,0.709157,0.0


## 6. Region-level score statistics

These are used as static features. They summarize each region's historical drought behavior.

In [7]:
global_score_mean = weekly_train[TARGET].mean()
global_score_median = weekly_train[TARGET].median()

region_stats = (
    weekly_train.dropna(subset=[TARGET])
    .groupby("region_id")[TARGET]
    .agg(
        region_score_mean="mean",
        region_score_median="median",
        region_score_std="std",
        region_score_min="min",
        region_score_max="max",
        region_score_count="count",
    )
    .reset_index()
)

region_high_freq = (
    weekly_train.dropna(subset=[TARGET])
    .assign(region_high_score=lambda x: (x[TARGET] >= 4).astype("float32"))
    .groupby("region_id")["region_high_score"]
    .mean()
    .reset_index()
)

region_stats = region_stats.merge(region_high_freq, on="region_id", how="left")
region_stats["region_score_std"] = region_stats["region_score_std"].fillna(0)

float_cols = region_stats.select_dtypes(include=["float64"]).columns
region_stats[float_cols] = region_stats[float_cols].astype("float32")

display(region_stats.head())

,region_id,region_score_mean,region_score_median,region_score_std,region_score_min,region_score_max,region_score_count,region_high_score
0,R1,0.997442,0.0,1.349347,0.0,5.0,782,0.072890
1,R1001,0.525575,0.0,1.038928,0.0,5.0,782,0.033248
2,R1002,0.369565,0.0,0.808245,0.0,4.0,782,0.005115
3,R1005,0.374680,0.0,0.823175,0.0,4.0,782,0.014066
4,R1006,0.341432,0.0,0.775619,0.0,4.0,782,0.005115


## 7. Build training samples

Each sample follows this structure:

```text
score history before input window + 13 weeks weather input → next 5 weekly scores
```

For a training anchor week `a`:

- Score history is known up to week `a`.
- Weather input is weeks `a+1` to `a+13`.
- Targets are scores from weeks `a+14` to `a+18`.

This matches the Kaggle test structure better than using score lags inside the 13-week input window.

In [8]:
weekly_feature_cols = [
    c for c in weekly_train.columns
    if c not in ["region_id", "week_idx", TARGET]
    and pd.api.types.is_numeric_dtype(weekly_train[c])
]

weather_sequence_cols = [c for c in weekly_feature_cols if c not in ["month", "month_sin", "month_cos"]]
season_cols = ["month", "month_sin", "month_cos"]
region_stat_cols = [c for c in region_stats.columns if c != "region_id"]
score_static_cols = [
    "last_known_score",
    "score_mean_4w",
    "score_mean_13w",
    "score_std_13w",
    "score_min_13w",
    "score_max_13w",
    "score_trend_4w_13w",
]

print("Sequence weather features:", len(weather_sequence_cols))
print(weather_sequence_cols[:20])
print("Static score features:", score_static_cols)
print("Region stat features:", region_stat_cols)

Sequence weather features: 60
['prec_mean', 'prec_std', 'prec_min', 'prec_max', 'prec_sum', 'humidity_mean', 'humidity_std', 'humidity_min', 'humidity_max', 'humidity_sum', 'tmp_mean', 'tmp_std', 'tmp_min', 'tmp_max', 'tmp_sum', 'dp_tmp_mean', 'dp_tmp_std', 'dp_tmp_min', 'dp_tmp_max', 'dp_tmp_sum']
Static score features: ['last_known_score', 'score_mean_4w', 'score_mean_13w', 'score_std_13w', 'score_min_13w', 'score_max_13w', 'score_trend_4w_13w']
Region stat features: ['region_score_mean', 'region_score_median', 'region_score_std', 'region_score_min', 'region_score_max', 'region_score_count', 'region_high_score']


In [9]:
def summarize_score_history(g, anchor_pos):
    past = g.iloc[:anchor_pos + 1][TARGET].dropna().astype(float)

    if len(past) == 0:
        return {
            "last_known_score": global_score_mean,
            "score_mean_4w": global_score_mean,
            "score_mean_13w": global_score_mean,
            "score_std_13w": 0.0,
            "score_min_13w": global_score_median,
            "score_max_13w": global_score_median,
            "score_trend_4w_13w": 0.0,
        }

    recent4 = past.tail(4)
    recent13 = past.tail(13)

    return {
        "last_known_score": past.iloc[-1],
        "score_mean_4w": recent4.mean(),
        "score_mean_13w": recent13.mean(),
        "score_std_13w": recent13.std() if len(recent13) > 1 else 0.0,
        "score_min_13w": recent13.min(),
        "score_max_13w": recent13.max(),
        "score_trend_4w_13w": recent4.mean() - recent13.mean(),
    }


def make_supervised_samples(weekly_df):
    seq_list = []
    static_list = []
    y_list = []
    meta_list = []

    stats_lookup = region_stats.set_index("region_id")

    for region_id, g in weekly_df.groupby("region_id", sort=False):
        g = g.sort_values("week_idx").reset_index(drop=True)
        n = len(g)

        # anchor_pos is the final week where score history is allowed.
        # input window: anchor_pos+1 : anchor_pos+1+INPUT_WEEKS
        # target window: after input window
        max_anchor_pos = n - INPUT_WEEKS - HORIZON - 1
        if max_anchor_pos < 0:
            continue

        for anchor_pos in range(max_anchor_pos + 1):
            input_start = anchor_pos + 1
            input_end = input_start + INPUT_WEEKS
            target_start = input_end
            target_end = target_start + HORIZON

            input_window = g.iloc[input_start:input_end]
            target_window = g.iloc[target_start:target_end]

            if len(input_window) != INPUT_WEEKS or len(target_window) != HORIZON:
                continue

            if target_window[TARGET].isna().any():
                continue

            seq = input_window[weather_sequence_cols + season_cols].values.astype("float32")

            score_feats = summarize_score_history(g, anchor_pos)
            static_feats = []
            static_feats += [score_feats[c] for c in score_static_cols]

            if region_id in stats_lookup.index:
                static_feats += stats_lookup.loc[region_id, region_stat_cols].values.astype("float32").tolist()
            else:
                static_feats += [global_score_mean, global_score_median, 0, global_score_median, global_score_median, 0, 0]

            y = target_window[TARGET].values.astype("float32")

            seq_list.append(seq)
            static_list.append(np.array(static_feats, dtype="float32"))
            y_list.append(y)
            meta_list.append((region_id, int(g.iloc[anchor_pos]["week_idx"])))

    X_seq = np.stack(seq_list).astype("float32")
    X_static = np.stack(static_list).astype("float32")
    y = np.stack(y_list).astype("float32")
    meta = pd.DataFrame(meta_list, columns=["region_id", "anchor_week"])

    return X_seq, X_static, y, meta

X_seq_all, X_static_all, y_all, meta_all = make_supervised_samples(weekly_train)

print("X_seq_all:", X_seq_all.shape)
print("X_static_all:", X_static_all.shape)
print("y_all:", y_all.shape)
print("meta_all:", meta_all.shape)

display(meta_all.head())

X_seq_all: (1717472, 13, 63)
X_static_all: (1717472, 14)
y_all: (1717472, 5)
meta_all: (1717472, 2)


,region_id,anchor_week
0,R1,0
1,R1,1
2,R1,2
3,R1,3
4,R1,4


## 8. Optional latest-N-years filtering

This uses `anchor_week`, not the unreliable calendar year.

In [10]:
if USE_LAST_N_YEARS:
    max_anchor = meta_all.groupby("region_id")["anchor_week"].transform("max")
    min_anchor = max_anchor - KEEP_LAST_N_WEEKS + 1
    keep_mask = (meta_all["anchor_week"] >= min_anchor).values

    X_seq_all = X_seq_all[keep_mask]
    X_static_all = X_static_all[keep_mask]
    y_all = y_all[keep_mask]
    meta_all = meta_all.loc[keep_mask].reset_index(drop=True)

    print(f"Using latest {N_YEARS_TO_USE} year(s), approximately {KEEP_LAST_N_WEEKS} weeks per region.")
else:
    print("Using full dataset.")

print("After filtering:")
print("X_seq_all:", X_seq_all.shape)
print("X_static_all:", X_static_all.shape)
print("y_all:", y_all.shape)

Using latest 2 year(s), approximately 104 weeks per region.
After filtering:
X_seq_all: (233792, 13, 63)
X_static_all: (233792, 14)
y_all: (233792, 5)


## 9. Train/validation split

The last few anchors per region are used for validation.

In [11]:
max_anchor = meta_all.groupby("region_id")["anchor_week"].transform("max")
val_start = max_anchor - VALIDATION_LAST_N_ANCHORS + 1
is_val = (meta_all["anchor_week"] >= val_start).values

train_idx = np.where(~is_val)[0]
val_idx = np.where(is_val)[0]

print("train samples:", len(train_idx))
print("val samples:", len(val_idx))

X_seq_train = X_seq_all[train_idx]
X_static_train = X_static_all[train_idx]
y_train = y_all[train_idx]

X_seq_val = X_seq_all[val_idx]
X_static_val = X_static_all[val_idx]
y_val = y_all[val_idx]

# Free full arrays if memory is tight
# Keep meta_all for reference only

gc.collect()

train samples: 222552
val samples: 11240


21

## 10. Normalize features

For neural networks, scaling is important. We standardize sequence and static features using training statistics only.
NaNs are replaced with zero after standardization.

In [12]:
seq_mean = np.nanmean(X_seq_train, axis=(0, 1), keepdims=True).astype("float32")
seq_std = np.nanstd(X_seq_train, axis=(0, 1), keepdims=True).astype("float32")
seq_std = np.where(seq_std < 1e-6, 1.0, seq_std).astype("float32")

static_mean = np.nanmean(X_static_train, axis=0, keepdims=True).astype("float32")
static_std = np.nanstd(X_static_train, axis=0, keepdims=True).astype("float32")
static_std = np.where(static_std < 1e-6, 1.0, static_std).astype("float32")

def normalize_seq(x):
    x = (x - seq_mean) / seq_std
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")

def normalize_static(x):
    x = (x - static_mean) / static_std
    return np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype("float32")

X_seq_train = normalize_seq(X_seq_train)
X_seq_val = normalize_seq(X_seq_val)

X_static_train = normalize_static(X_static_train)
X_static_val = normalize_static(X_static_val)

print("Any NaN seq train:", np.isnan(X_seq_train).any())
print("Any NaN static train:", np.isnan(X_static_train).any())

Any NaN seq train: False
Any NaN static train: False


## 11. PyTorch dataset and model

In [13]:
class DroughtDataset(Dataset):
    def __init__(self, x_seq, x_static, y=None):
        self.x_seq = torch.tensor(x_seq, dtype=torch.float32)
        self.x_static = torch.tensor(x_static, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x_seq)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x_seq[idx], self.x_static[idx]
        return self.x_seq[idx], self.x_static[idx], self.y[idx]


class CNN1DRegressor(nn.Module):
    def __init__(self, n_seq_features, n_static_features, horizon=5):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(n_seq_features, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.static_net = nn.Sequential(
            nn.Linear(n_static_features, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
        )

        self.head = nn.Sequential(
            nn.Linear(32 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, horizon),
        )

    def forward(self, x_seq, x_static):
        # input x_seq: batch, time, features
        x_seq = x_seq.transpose(1, 2)  # batch, features, time
        x = self.conv(x_seq)
        x = self.pool(x).squeeze(-1)

        s = self.static_net(x_static)
        out = self.head(torch.cat([x, s], dim=1))
        return out


train_loader = DataLoader(
    DroughtDataset(X_seq_train, X_static_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    DroughtDataset(X_seq_val, X_static_val, y_val),
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

model = CNN1DRegressor(
    n_seq_features=X_seq_train.shape[2],
    n_static_features=X_static_train.shape[1],
    horizon=HORIZON,
).to(device)

print(model)

CNN1DRegressor(
  (conv): Sequential(
    (0): Conv1d(63, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Conv1d(64, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (pool): AdaptiveAvgPool1d(output_size=1)
  (static_net): Sequential(
    (0): Linear(in_features=14, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64

## 12. Train the 1D CNN

In [14]:
criterion = nn.L1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

best_val_mae = float("inf")
best_state = None
bad_epochs = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for xb_seq, xb_static, yb in train_loader:
        xb_seq = xb_seq.to(device, non_blocking=True)
        xb_static = xb_static.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        preds = model(xb_seq, xb_static)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for xb_seq, xb_static, yb in val_loader:
            xb_seq = xb_seq.to(device, non_blocking=True)
            xb_static = xb_static.to(device, non_blocking=True)
            preds = model(xb_seq, xb_static)
            preds = torch.clamp(preds, 0, 5)

            val_preds.append(preds.cpu().numpy())
            val_targets.append(yb.numpy())

    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)

    val_mae = mean_absolute_error(val_targets, val_preds)
    scheduler.step(val_mae)

    print(f"Epoch {epoch:02d} | train MAE loss {np.mean(train_losses):.5f} | val MAE {val_mae:.5f}")

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

print("Best validation MAE:", best_val_mae)
model.load_state_dict(best_state)

Epoch 01 | train MAE loss 0.38947 | val MAE 0.16077
Epoch 02 | train MAE loss 0.32012 | val MAE 0.15971
Epoch 03 | train MAE loss 0.30038 | val MAE 0.16843
Epoch 04 | train MAE loss 0.28605 | val MAE 0.14239
Epoch 05 | train MAE loss 0.27779 | val MAE 0.15245
Epoch 06 | train MAE loss 0.26974 | val MAE 0.14987
Epoch 07 | train MAE loss 0.26536 | val MAE 0.14965
Epoch 08 | train MAE loss 0.25547 | val MAE 0.15847
Epoch 09 | train MAE loss 0.25205 | val MAE 0.14517
Early stopping.
Best validation MAE: 0.14238667488098145


<All keys matched successfully>

In [15]:
# Detailed validation MAE by horizon
model.eval()
val_preds = []
with torch.no_grad():
    for xb_seq, xb_static, yb in val_loader:
        xb_seq = xb_seq.to(device, non_blocking=True)
        xb_static = xb_static.to(device, non_blocking=True)
        preds = torch.clamp(model(xb_seq, xb_static), 0, 5)
        val_preds.append(preds.cpu().numpy())

val_preds = np.vstack(val_preds)

print("Overall validation MAE:", mean_absolute_error(y_val, val_preds))
for h in range(HORIZON):
    print(f"week{h+1} MAE:", mean_absolute_error(y_val[:, h], val_preds[:, h]))

Overall validation MAE: 0.14238667488098145
week1 MAE: 0.14439675211906433
week2 MAE: 0.14089520275592804
week3 MAE: 0.13906295597553253
week4 MAE: 0.14198565483093262
week5 MAE: 0.14561626315116882


## 13. Build test samples

For test:

- Sequence input = the 13 weekly rows from `test.csv`.
- Static score history = latest known training score/history for that region.
- Region statistics = full training history.

In [16]:
def make_test_samples(weekly_test_df, weekly_train_df):
    seq_list = []
    static_list = []
    region_list = []

    stats_lookup = region_stats.set_index("region_id")

    for region_id, g_test in weekly_test_df.groupby("region_id", sort=False):
        g_test = g_test.sort_values("week_idx").reset_index(drop=True)
        input_window = g_test.tail(INPUT_WEEKS)

        if len(input_window) != INPUT_WEEKS:
            print("Warning: test region does not have 13 weeks:", region_id, len(input_window))
            continue

        seq = input_window[weather_sequence_cols + season_cols].values.astype("float32")

        g_train_region = weekly_train_df[weekly_train_df["region_id"] == region_id].sort_values("week_idx").reset_index(drop=True)
        anchor_pos = len(g_train_region) - 1
        score_feats = summarize_score_history(g_train_region, anchor_pos)

        static_feats = []
        static_feats += [score_feats[c] for c in score_static_cols]

        if region_id in stats_lookup.index:
            static_feats += stats_lookup.loc[region_id, region_stat_cols].values.astype("float32").tolist()
        else:
            static_feats += [global_score_mean, global_score_median, 0, global_score_median, global_score_median, 0, 0]

        seq_list.append(seq)
        static_list.append(np.array(static_feats, dtype="float32"))
        region_list.append(region_id)

    X_seq_test = np.stack(seq_list).astype("float32")
    X_static_test = np.stack(static_list).astype("float32")
    regions = pd.Series(region_list, name="region_id")

    return X_seq_test, X_static_test, regions

X_seq_test, X_static_test, test_regions = make_test_samples(weekly_test, weekly_train)

X_seq_test = normalize_seq(X_seq_test)
X_static_test = normalize_static(X_static_test)

print("X_seq_test:", X_seq_test.shape)
print("X_static_test:", X_static_test.shape)
print("test_regions:", test_regions.shape)

X_seq_test: (2248, 13, 63)
X_static_test: (2248, 14)
test_regions: (2248,)


## 14. Predict test and create submission

In [17]:
test_loader = DataLoader(
    DroughtDataset(X_seq_test, X_static_test, y=None),
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

model.eval()
test_preds = []

with torch.no_grad():
    for xb_seq, xb_static in test_loader:
        xb_seq = xb_seq.to(device, non_blocking=True)
        xb_static = xb_static.to(device, non_blocking=True)
        preds = torch.clamp(model(xb_seq, xb_static), 0, 5)
        test_preds.append(preds.cpu().numpy())

test_preds = np.vstack(test_preds).astype("float32")

print("Prediction stats")
print("min:", float(test_preds.min()))
print("max:", float(test_preds.max()))
print("mean:", float(test_preds.mean()))

pred_cols = [c for c in sample_submission.columns if c != "region_id"]

display(pd.DataFrame(test_preds, columns=pred_cols[:HORIZON]).head())

Prediction stats
min: 0.0
max: 5.0
mean: 0.5950314402580261


,pred_week1,pred_week2,pred_week3,pred_week4,pred_week5
0,0.761044,0.919296,1.074628,1.262393,1.386121
1,0.000184,0.000000,0.000035,0.000000,0.000045
2,0.000174,0.000000,0.000031,0.000000,0.000035
3,0.323362,0.315735,0.294685,0.266159,0.226365
4,0.893698,0.869352,0.820148,0.754543,0.676345


In [18]:
pred_df = pd.DataFrame(test_preds, columns=pred_cols[:HORIZON])
pred_df["region_id"] = test_regions.values

submission = sample_submission.copy()
submission = submission.drop(columns=pred_cols[:HORIZON]).merge(pred_df, on="region_id", how="left")
submission = submission[sample_submission.columns]

assert not submission[pred_cols[:HORIZON]].isna().any().any(), "Submission contains NaNs."

out_name = f"submission_1dcnn_last_{N_YEARS_TO_USE}years.csv" if USE_LAST_N_YEARS else "submission_1dcnn_full_data.csv"
out_path = os.path.join(OUT_DIR, out_name)
submission.to_csv(out_path, index=False)

print("Saved submission to:", out_path)
display(submission.head())

Saved submission to: /kaggle/working/cnn_1d_outputs/submission_1dcnn_last_2years.csv


,region_id,pred_week1,pred_week2,pred_week3,pred_week4,pred_week5
0,R1,0.761044,0.919296,1.074628,1.262393,1.386121
1,R2,0.000174,0.000000,0.000031,0.000000,0.000035
2,R3,0.000174,0.000000,0.000031,0.000000,0.000035
3,R4,0.147107,0.149197,0.152671,0.142755,0.135466
4,R6,0.576888,0.603938,0.616154,0.633417,0.638658


## Notes

This is meant as a **fast first 1D CNN baseline**, not a fully tuned neural network.

Good next experiments:

- Increase `MAX_DAILY_WEATHER_FEATURES` from 12 to 20.
- Try `N_YEARS_TO_USE = 3` or `5`.
- Increase filters from 64 to 128.
- Add a region embedding instead of only region statistics.
- Compare against LightGBM using the same validation setup.